# Learned Stage-Wise Curvature Tuning on FGVC Aircraft

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the FGVC-Aircraft dataset using an ImageNet-pretrained ResNet-50.

FGVC-Aircraft is a fine-grained image-classification dataset in which the model must distinguish between visually similar aircraft categories. This makes it useful for testing whether stage-wise curvature adaptation can help when the downstream task requires subtle feature discrimination.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization changes accuracy or the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Sat Oct  3 20:36:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b stage-wise-ct \
    https://github.com/Syed1611/curvature-tuning-research.git

%cd /content/curvature-tuning-research/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...


remote: Enumerating objects: 575, done.
remote: Counting objects: 100% (247/247), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 575 (delta 153), reused 181 (delta 103), pack-reused 328 (from 1)
Receiving objects: 100% (575/575), 6.93 MiB | 5.68 MiB/s, done.
Resolving deltas: 100% (284/284), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
068c466 (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) Updated aircraft with complete results


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 73.5 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 85.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
from pathlib import Path

data_file = Path(
    "/content/curvature-tuning-research/src/curvature-tuning/utils/data.py"
)

text = data_file.read_text(encoding="utf-8")

old = '''    elif dataset_to_use == 'fgvc-aircraft':
        hf_dataset = datasets.load_dataset(
            "HuggingFaceM4/FGVC-Aircraft",
            num_proc=2
        )

        hf_trainset = hf_dataset["train"]
        hf_valset = hf_dataset["validation"]
        hf_testset = hf_dataset["test"]

        train_set = HuggingFaceDataset(
            hf_trainset,
            transform=transform_train
        )

        val_set = HuggingFaceDataset(
            hf_valset,
            transform=transform_test
        )

        test_set = HuggingFaceDataset(
            hf_testset,
            transform=transform_test
        )
'''

new = '''    elif dataset_to_use == 'fgvc-aircraft':
        train_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="train",
            annotation_level="variant",
            transform=transform_train,
            download=True,
        )

        val_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="val",
            annotation_level="variant",
            transform=transform_test,
            download=True,
        )

        test_set = torchvision.datasets.FGVCAircraft(
            root="./data",
            split="test",
            annotation_level="variant",
            transform=transform_test,
            download=True,
        )
'''

if new in text:
    print("FGVC-Aircraft patch already applied.")
elif old in text:
    text = text.replace(old, new)
    data_file.write_text(text, encoding="utf-8")
    print("FGVC-Aircraft soft patch applied.")
else:
    print("Expected FGVC-Aircraft block was not found. Nothing changed.")

FGVC-Aircraft soft patch applied.


In [5]:
%cd /content/curvature-tuning-research/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## FGVC Aircraft Dataset

Before running the experiments, the FGVC Aircraft data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [6]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_fgvc-aircraft",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, FGVC Aircraft dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [4]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 09:00:12.029 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs20.log
2026-10-03 09:00:12.034 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 184MB/s]
2026-10-03 09:00:15.017 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 09:00:15.084 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 09:00:15.084 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 09:01:31.463 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.724136, Accuracy: 3.12%
2026-10-03 09:01:31.809 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.463026, Accuracy: 4.95%
2026-10-03 09:01:32.067 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.258730, 

In [5]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 09:52:15.389 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs20.log
2026-10-03 09:52:15.392 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 09:52:16.278 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 09:52:16.336 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 09:52:16.336 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 09:53:38.802 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.714942, Accuracy: 0.00%
2026-10-03 09:53:39.187 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.500874, Accuracy: 3.40%
2026-10-03 09:53:39.472 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.176976, Val Accuracy: 8.22%
2026-10-03 09:53:39.473 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.22 at epoch 1
2026-10-03 09:53:39.550 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.0

In [6]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 10:47:19.376 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs20.log
2026-10-03 10:47:19.380 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 10:47:20.389 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 10:47:20.488 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 10:47:20.488 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 10:48:43.529 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.676371, Accuracy: 0.00%
2026-10-03 10:48:43.867 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.477705, Accuracy: 3.53%
2026-10-03 10:48:44.139 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.140138, Val Accuracy: 8.22%
2026-10-03 10:48:44.141 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.22 at epoch 1
2026-10-03 10:48:44.225 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.9

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 11:42:48.828 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 11:42:48.905 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 11:42:49.924 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 11:42:49.925 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 11:42:49.985 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 11:42:49.986 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 11:42:49.986 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 11:42:52.344 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.624586, Accuracy: 0.00%
2026-10-03 11:43:48.558 | INFO     | train:train_

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 12:18:33.885 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 12:18:33.980 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 12:18:34.942 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 12:18:34.942 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 12:18:35.031 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 12:18:35.033 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 12:18:35.033 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 12:18:37.362 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.717534, Accuracy: 0.00%
2026-10-03 12:19:34.209 | INFO     | train:train_

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 12:54:01.600 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-03 12:54:01.674 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 12:54:02.669 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 12:54:02.670 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 12:54:02.762 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 12:54:02.763 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 12:54:02.764 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 12:54:05.006 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.663696, Accuracy: 0.00%
2026-10-03 12:55:01.773 | INFO     | train:train_

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [10]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 13:29:36.139 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-03 13:29:36.209 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 13:29:37.134 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 13:29:37.135 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 13:29:37.257 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 13:29:37.258 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 13:29:37.259 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 13:29:39.482 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.531807, Accuracy: 0.00%
2026-10-03 13:30:36.124 | INFO     | 

In [11]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 14:03:40.143 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-03 14:03:40.202 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 14:03:41.010 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 14:03:41.010 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 14:03:41.073 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 14:03:41.074 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 14:03:41.074 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 14:03:42.940 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.726221, Accuracy: 0.00%
2026-10-03 14:04:40.831 | INFO     

In [12]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-03 14:36:37.105 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-03 14:36:37.199 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 14:36:38.009 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 14:36:38.010 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 14:36:38.084 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 14:36:38.085 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 14:36:38.086 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 14:36:40.208 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.675265, Accuracy: 0.00%
2026-10-03 14:37:38.039 | INFO     

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [13]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 15:11:29.809 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs30.log
2026-10-03 15:11:29.813 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 15:11:30.760 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 15:11:30.809 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 15:11:30.809 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 15:12:46.407 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.724136, Accuracy: 3.12%
2026-10-03 15:12:46.756 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.463026, Accuracy: 4.95%
2026-10-03 15:12:47.014 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.258730, Val Accuracy: 6.33%
2026-10-03 15:12:47.015 | INFO     | train:linear_probe:190 - New best validation accuracy: 6.33 at epoch 1
2026-10-03 15:12:47.090 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.8

In [14]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 16:07:12.105 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs30.log
2026-10-03 16:07:12.108 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 16:07:13.102 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 16:07:13.166 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 16:07:13.166 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 16:08:28.106 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.714942, Accuracy: 0.00%
2026-10-03 16:08:28.493 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.500874, Accuracy: 3.40%
2026-10-03 16:08:28.786 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.176976, Val Accuracy: 8.22%
2026-10-03 16:08:28.787 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.22 at epoch 1
2026-10-03 16:08:28.877 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 4.0

In [15]:
!WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-03 17:02:13.142 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs30.log
2026-10-03 17:02:13.147 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 17:02:14.098 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 17:02:14.152 | INFO     | __main__:main:116 - Number of trainable parameters: 204900
2026-10-03 17:02:14.152 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 17:03:28.393 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.676371, Accuracy: 0.00%
2026-10-03 17:03:28.758 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 4.477705, Accuracy: 3.53%
2026-10-03 17:03:29.022 | INFO     | train:test_epoch:90 - Epoch 1, Val Loss: 4.140138, Val Accuracy: 8.22%
2026-10-03 17:03:29.024 | INFO     | train:linear_probe:190 - New best validation accuracy: 8.22 at epoch 1
2026-10-03 17:03:29.139 | INFO     | train:train_epoch:47 - Epoch 2, Step 0, Loss: 3.9

In [16]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 17:56:22.196 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 17:56:22.263 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 17:56:23.105 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 17:56:23.106 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 17:56:23.174 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 17:56:23.175 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 17:56:23.175 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 17:56:25.403 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.624586, Accuracy: 0.00%
2026-10-03 17:57:21.301 | INFO     | train:train_

In [17]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 18:46:44.592 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 18:46:44.677 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 18:46:45.496 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 18:46:45.496 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 18:46:45.566 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 18:46:45.567 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 18:46:45.568 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 18:46:47.699 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.717534, Accuracy: 0.00%
2026-10-03 18:47:44.624 | INFO     | train:train_

In [18]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 19:37:02.417 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-03 19:37:02.481 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 19:37:03.326 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-03 19:37:03.326 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 19:37:03.392 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-03 19:37:03.393 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 19:37:03.394 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 19:37:05.483 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.663696, Accuracy: 0.00%
2026-10-03 19:38:01.997 | INFO     | train:train_

In [7]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 20:45:26.156 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-03 20:45:26.219 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 219MB/s]
100%|██████████████████████████████████████| 2.75G/2.75G [01:22<00:00, 33.6MB/s]
2026-10-03 20:47:04.468 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 20:47:04.468 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 20:47:13.401 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-03 20:47:13.402 | INFO    

In [8]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 21:37:45.339 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-03 21:37:45.434 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 21:37:46.257 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 21:37:46.258 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 21:37:46.321 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-03 21:37:46.322 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 21:37:46.322 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 21:37:48.494 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.726221, Accuracy: 0.00%
2026-10-03 21:38:45.309 | INFO     

In [9]:
!WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds fgvc-aircraft \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-03 22:29:33.744 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_fgvc-aircraft_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-03 22:29:33.823 | INFO     | __main__:main:302 - Running on cuda
2026-10-03 22:29:34.809 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-03 22:29:34.809 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-03 22:29:34.912 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-03 22:29:34.913 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-03 22:29:34.914 | INFO     | __main__:main:439 - Total trainable parameters: 204904
2026-10-03 22:29:37.182 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 4.675265, Accuracy: 0.00%
2026-10-03 22:30:34.542 | INFO     